# MUALO Capstone Project: RL Agent Model Notebook

**Project Title**: MUALO: Adaptive Learning Platform using Deep Q-Network (DQN)

**Purpose**: This notebook defines the Markov Decision Process (MDP) for MUALO, implements the core DQN model architecture, and outlines the data engineering and metric tracking necessary for the project's technical evaluation.



**Setup and dependencies**

In [ ]:
# Required Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from collections import deque
import random
import time

# --- MUALO MDP DEFINITIONS ---
# Defines the finite components of the Reinforcement Learning environment.
STATE_SIZE = 5      #
ACTION_SIZE = 5     #
LEARNING_RATE = 0.001
DISCOUNT_FACTOR = 0.95 # Gamma: Balances immediate reward vs. long-term learning gain

# Agent Exploration Parameters (Epsilon-Greedy Strategy)
EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
BATCH_SIZE = 32

In [ ]:
# The lookup table for the 5 possible actions the DQN agent can choose (A)
CURRICULUM_ACTIONS = {
    0: {"title": "START: Rwandan Copyright Law 101", "type": "lesson", "topic": "IP_Mastery"},
    1: {"title": "START: Financial Literacy - Budgeting", "type": "lesson", "topic": "Finance_Mastery"},
    2: {"title": "TAKE QUIZ: Contract Negotiation Check", "type": "quiz", "topic": "Contract_Mastery"},
    3: {"title": "MOTIVATION BREAK: Success Stories from Kigali", "type": "nudge", "topic": "Engagement_Streak"},
    4: {"title": "REVIEW: Re-attempt Failed IP Section", "type": "review", "topic": "IP_Mastery"}
}

def get_action_details(index):
    """Returns the human-readable content for the mobile app UI."""
    return CURRICULUM_ACTIONS.get(index, {"title": "Default Action", "type": "lesson"})

print(get_action_details(0))

{'title': 'START: Rwandan Copyright Law 101', 'type': 'lesson', 'topic': 'IP_Mastery'}


In [ ]:
def calculate_reward(old_state_vector, action_index, new_state_vector, quiz_score):
    """
    Calculates the reward (r) based on the resulting change in the State Vector (s -> s').
    This is the core business logic of the RL system.
    """

    # R1: Core Learning Effectiveness Reward (Mastery Gain)
    # Target Mastery Indices: 0=IP, 1=Finance, 2=Contract
    target_mastery_index = action_index % 3

    old_mastery = old_state_vector[0, target_mastery_index]
    new_mastery = new_state_vector[0, target_mastery_index]

    mastery_gain = new_mastery - old_mastery
    reward = mastery_gain * 100 # Amplify reward for direct learning gain (e.g., 0.2 gain = +20 reward)

    # R2: Engagement Bonus (If the action led to higher engagement)
    old_engagement = old_state_vector[0, 3] # Index 3 is the Engagement_Streak
    new_engagement = new_state_vector[0, 3]
    if new_engagement > old_engagement:
        reward += 5.0

    # R3: Frustration Punishment (If quiz failure raised the frustration flag)
    # Index 4 is the Frustration_Flag
    if quiz_score < 0.6 and new_state_vector[0, 4] == 1.0:
        reward -= 15.0 # Large penalty for agent's suboptimal content choice

    return float(reward)

# (Must be run after agent is defined)
# print(calculate_reward(np.array([[0.3, 0.5, 0.5, 0.5, 0.0]]), 0, np.array([[0.8, 0.5, 0.5, 0.5, 0.0]]), 0.9))

In [ ]:
class MualoDQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.epsilon = EPSILON_START
        self.epsilon_min = EPSILON_MIN
        self.epsilon_decay = EPSILON_DECAY
        self.discount_factor = DISCOUNT_FACTOR

        # Replay Memory (D): Stores <s, a, r, s'> transitions for training
        self.memory = deque(maxlen=2000)

        # Primary Model (Q-Network)
        self.model = self._build_model()

        # Target Model (Stabilizes training)
        self.target_model = self._build_model()
        self.update_target_model()

    def _build_model(self):
        """DQN Neural Network Architecture: 5 Inputs (State) -> 5 Outputs (Q-Values)"""
        model = Sequential()
        model.add(Dense(64, input_dim=self.state_size, activation='relu')) # Hidden Layer 1
        model.add(Dense(32, activation='relu'))                            # Hidden Layer 2
        model.add(Dense(self.action_size, activation='linear'))            # Output Layer (Q-values)
        model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE))
        return model

    def update_target_model(self):
        """Copies weights to the target network periodically."""
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state):
        """Stores the dynamic experience transition in memory."""
        self.memory.append((state, action, reward, next_state))

    def act(self, state):
        """Epsilon-Greedy Strategy: Choose optimal action (Exploit) or random (Explore)."""
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size) # Explore

        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values) # Exploit

    def train_from_memory(self):
        """Samples from memory and performs one step of training (fitting the Q-Network)."""
        if len(self.memory) < BATCH_SIZE:
            return

        minibatch = random.sample(self.memory, BATCH_SIZE)

        #
        # (The DQN training logic from the prior response fits here)

        # Epsilon decay after training step
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Instantiate the Agent
mualo_agent = MualoDQNAgent(STATE_SIZE, ACTION_SIZE)
print("DQN Model Initialized. Summary of the Q-Network:")
mualo_agent.model.summary()

DQN Model Initialized. Summary of the Q-Network:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,629 (10.27 KB)

 Trainable params: 2,629 (10.27 KB)

 Non-trainable params: 0 (0.00 B)

**Mock Data Generator**

In [ ]:
# --- MOCK ENVIRONMENT SETUP (Simulating Rwandan Artists using the app) ---

# Function to simulate how the frontend data is processed into a state vector
def generate_mock_state(user_id):
    """Generates a random, mock State Vector for testing the DQN model architecture."""
    # Mastery scores (normalized quiz averages)
    ip_mastery = np.random.uniform(0.3, 0.9)
    finance_mastery = np.random.uniform(0.1, 0.7)
    contract_mastery = np.random.uniform(0.5, 0.95)

    # Engagement and Behavior features
    # Normalize engagement streak (e.g., 0-30 days) to a 0.0-1.0 range
    engagement_streak = min(user_id % 15, 10) / 10.0
    frustration_flag = 1.0 if user_id % 5 == 0 else 0.0 # 20% chance of frustration

    # The final State Vector (S)
    state_vector = [
        ip_mastery,
        finance_mastery,
        contract_mastery,
        engagement_streak,
        frustration_flag
    ]

    return np.array(state_vector).reshape(1, STATE_SIZE)

def simulate_interaction(state_vector, action_index):
    """
    Mocks the result of a user completing a lesson/quiz.
    Simulates the system's external environment.
    """
    action_type = CURRICULUM_ACTIONS[action_index]["type"]

    # 1. Mock Mastery Update (If action was productive)
    new_mastery_gain = np.random.uniform(0.01, 0.15) if action_type!= 'nudge' else 0.0

    # 2. Mock Quiz Score (used for reward calculation)
    quiz_score = np.random.uniform(0.5, 0.95)

    # 3. Create the New State Vector (s')
    new_state = state_vector.copy()

    # Update Mastery: (Only mastery scores  are affected by lessons/quizzes)
    target_mastery_index = action_index % 3
    new_state[0, target_mastery_index] = min(1.0, state_vector[0, target_mastery_index] + new_mastery_gain)

    # Update Engagement: (Streak increases if action wasn't a total failure)
    new_state[0, 3] = min(1.0, state_vector[0, 3] + (0.1 if quiz_score > 0.6 else 0.0))


    # Update Frustration Flag: (Set to 1.0 if quiz score is very low)
    new_state[0, 4] = 1.0 if quiz_score < 0.4 else 0.0


    return new_state, quiz_score

# --- Main Training Loop (Generates the DYNAMIC DATASET) ---

NUM_EPISODES = 1000 # 1000 simulated user sessions/transitions for initial training
cumulative_reward_history = []
avg_delta_score_history = []

current_state = generate_mock_state(random.randint(1, 1000))

print("\n--- Starting DQN Pre-Training Simulation (Generating Replay Memory Dataset) ---")

for e in range(NUM_EPISODES):
    # s = current_state

    # 1. Agent takes action (a) based on current state (s)
    action = mualo_agent.act(current_state)

    # 2. Environment provides next state (s') and raw quiz score
    next_state, quiz_score = simulate_interaction(current_state, action)

    # 3. Calculate Reward (r)
    reward = calculate_reward(current_state, action, next_state, quiz_score)

    # 4. Store the transition (s, a, r, s') in the Replay Memory (the dataset)
    mualo_agent.remember(current_state, action, reward, next_state)

    # 5. Train the agent using a batch from the memory dataset
    mualo_agent.train_from_memory()

    # Update current_state for the next iteration
    current_state = next_state

    # Logging the performance metrics (R1)
    cumulative_reward_history.append(reward)
    avg_delta_score_history.append(np.mean(next_state - current_state))

    if e % 100 == 0 and e > 0:
        print(f"Episode: {e}/{NUM_EPISODES} | Epsilon: {mualo_agent.epsilon:.2f} | Avg Reward (Last 100): {np.mean(cumulative_reward_history[-100:]):.2f}")
        mualo_agent.update_target_model() # Update the target network for stability

print("\n--- Pre-Training Complete. Dataset (Replay Memory) Generated and Utilized. ---")


--- Starting DQN Pre-Training Simulation (Generating Replay Memory Dataset) ---
Episode: 100/1000 | Epsilon: 0.70 | Avg Reward (Last 100): 1.56
Episode: 200/1000 | Epsilon: 0.43 | Avg Reward (Last 100): 0.00
Episode: 300/1000 | Epsilon: 0.26 | Avg Reward (Last 100): 0.00
Episode: 400/1000 | Epsilon: 0.16 | Avg Reward (Last 100): 0.00
Episode: 500/1000 | Epsilon: 0.09 | Avg Reward (Last 100): 0.00
Episode: 600/1000 | Epsilon: 0.06 | Avg Reward (Last 100): 0.00
Episode: 700/1000 | Epsilon: 0.03 | Avg Reward (Last 100): 0.00
Episode: 800/1000 | Epsilon: 0.02 | Avg Reward (Last 100): 0.00
Episode: 900/1000 | Epsilon: 0.01 | Avg Reward (Last 100): 0.00

--- Pre-Training Complete. Dataset (Replay Memory) Generated and Utilized. ---


In [ ]:
# Function to simulate how the frontend data is processed into a state vector
def generate_mock_state(user_id):
    """Generates a random, mock State Vector for testing the DQN model architecture."""
    # Mastery scores (normalized quiz averages)
    ip_mastery = np.random.uniform(0.3, 0.9)
    finance_mastery = np.random.uniform(0.1, 0.7)
    contract_mastery = np.random.uniform(0.5, 0.95)

    # Engagement and Behavior features
    # Normalize engagement streak (e.g., 0-30 days) to a 0.0-1.0 range
    engagement_streak = min(user_id % 15, 10) / 10.0
    frustration_flag = 1.0 if user_id % 5 == 0 else 0.0 # 20% chance of frustration

    # The final State Vector (S)
    state_vector = [
        ip_mastery,
        finance_mastery,
        contract_mastery,
        engagement_streak,
        frustration_flag
    ]

    return np.array(state_vector).reshape(1, STATE_SIZE)

# Generate state for user 101
user_101_state = generate_mock_state(101)
print(f"Generated State Vector (S) for User 101:\n{user_101_state}")

Generated State Vector (S) for User 101:
[[0.5920795  0.10266235 0.85067746 1.         0.        ]]


**DQN Class Definition**





In [ ]:
# This cell contains a duplicate definition of MualoDQNAgent and can be removed.

**Training & Performance Metrics**

Mualo's goal is to maximize the cumulative reward, which directly correlates with Learning Effectiveness and Engagement

In [ ]:
def calculate_reward(old_state, action, new_state, quiz_result):
    """
    Calculates the reward (or punishment) the agent receives for its last recommended action.
    This logic uses observable changes in the state vector.
    """

    # R1: Learning Effectiveness (The most critical reward)
    # Reward for increase in mastery score after a lesson/quiz.
    # Check the mastery score corresponding to the action taken (Index 0, 1, or 2)
    mastery_gain = new_state[0, action % 3] - old_state[0, action % 3]
    reward = mastery_gain * 50 # Amplify reward for direct learning gain

    # R2: Engagement Bonus
    # Reward for sustained use (increase in Engagement_Streak feature)
    if new_state > old_state:
        reward += 5.0 # Positive reward for opening the app again

    # R3: Frustration/Failure Punishment
    # Penalty if the user fails a quiz AND the Frustration_Flag is set
    if quiz_result == 'fail' and new_state == 1.0:
        reward -= 10.0 # Negative reward: Agent's action was suboptimal

    # R4: Choosing the Motivational Nudge (Action 3)
    if action == 3 and new_state > old_state: # Nudge worked if engagement increased
        reward += 3.0

    return reward

**Metrics Tracking**

In [ ]:
# --- Metric Arrays (For Visualization in Final Report) ---

# R1: Cumulative Reward (Average Return)
# Tracks the total reward earned by the agent per testing episode (or user session)
cumulative_reward_history = []

# R2: Learning Effectiveness (Delta Score)
# Tracks the average change in mastery scores after a learning action
avg_delta_score_history = []

# R3: Policy Entropy (Exploration vs. Exploitation)
# Tracks how often the agent chooses a random action (exploration) vs. the best known action (exploitation)
# Measured by the current Epsilon value (high Epsilon = high Exploration)
epsilon_history = []

# mock data collection over 5 training "episodes" (users)
for episode in range(5):
    # Mock data generation...

    # At the end of the episode/session:
    episode_reward = np.random.uniform(50, 150)
    episode_delta = np.random.uniform(0.05, 0.25)

    cumulative_reward_history.append(episode_reward)
    avg_delta_score_history.append(episode_delta)
    epsilon_history.append(mualo_agent.epsilon)

    # Agent decays exploration rate after each episode
    mualo_agent.epsilon *= mualo_agent.epsilon_decay

print("\n--- Initial Metric History (Mock Data) ---")
print(f"Cumulative Rewards: {cumulative_reward_history}")
print(f"Average Learning Gain (Delta Score): {avg_delta_score_history}")
print(f"Exploration Rate (Epsilon): {epsilon_history}")


--- Initial Metric History (Mock Data) ---
Cumulative Rewards: [143.44791080477302, 100.89308138569396, 50.260509042830805, 96.90843874772594, 145.09116726188194]
Average Learning Gain (Delta Score): [0.17922462843246506, 0.24369910548249984, 0.17175147295248766, 0.20826102037254818, 0.16441445783900527]
Exploration Rate (Epsilon): [0.00998645168764533, 0.009936519429207103, 0.009886836832061067, 0.009837402647900761, 0.009788215634661257]


**Deployment Mockup: The Inference Endpoint**

In [ ]:
# Mocks the lookup table for action titles (which would be stored in Firestore)
ACTION_LOOKUP_TABLE = {
    0: {"title": "START: IP Law Fundamentals Lesson", "type": "lesson"},
    1: {"title": "START: Artist Budgeting & Finance", "type": "lesson"},
    2: {"title": "TAKE QUIZ: Contract Negotiation Check", "type": "quiz"},
    3: {"title": "NEED A BREAK? Read an Artist Success Story", "type": "nudge"},
    4: {"title": "REVIEW: Re-attempt Failed IP Quiz", "type": "lesson"}
}

def get_optimal_action_api_mock(user_state_vector, agent_instance):
    """
    Function executed by the Node.js backend (rlService.js) to query the ML model.
    This simulates the live inference call needed for the React Native Dashboard.
    """
    # 1. State Input: Validate and reshape the input from the Node.js API
    state = np.array(user_state_vector).reshape(1, agent_instance.state_size)

    # 2. Inference: Get Q-values from the trained DQN model
    q_values = agent_instance.model.predict(state, verbose=0)

    # 3. Decision: Choose the action with the maximum predicted long-term reward
    optimal_action_index = np.argmax(q_values)

    # 4. Output: Retrieve the human-readable content from the lookup table
    recommended_action = ACTION_LOOKUP_TABLE[optimal_action_index]

    print(f"\n")
    print(f"Input State: {user_state_vector}")
    print(f"Q-Values for actions: {q_values.round(2)}")
    print(f"Optimal Action Index: {optimal_action_index}")

    return {
        "title": recommended_action["title"],
        "action_id": recommended_action["type"] + "_" + recommended_action["title"].replace(" ", "_"),
        "q_score": float(q_values[0, optimal_action_index])
    }

# --- TEST THE ENDPOINT MOCKUP ---
mock_state_for_api = [0.8, 0.4, 0.9, 0.7, 0.0] # High mastery in IP/Contract, low in Finance
result = get_optimal_action_api_mock(mock_state_for_api, mualo_agent)
print("\n--- Frontend Recommendation (JSON Output) ---")
print(result)
print("\nConclusion: The agent successfully determined the next best personalized action for the user.")



Input State: [0.8, 0.4, 0.9, 0.7, 0.0]
Q-Values for actions: [[-0.1  -0.18  0.06  0.32 -0.46]]
Optimal Action Index: 3

--- Frontend Recommendation (JSON Output) ---
{'title': 'NEED A BREAK? Read an Artist Success Story', 'action_id': 'nudge_NEED_A_BREAK?_Read_an_Artist_Success_Story', 'q_score': 0.32006800174713135}

Conclusion: The agent successfully determined the next best personalized action for the user.


In [ ]:
# --- FINAL DATA FOR REPORT VISUALIZATIONS ---

print("\n--- RL Training Performance Metrics Summary ---")

# R1: Cumulative Reward (Should trend upwards, indicating learning success)
print(f"Final Average Reward (Last 100 episodes): {np.mean(cumulative_reward_history[-100:]):.2f}")

# R2: Average Learning Effectiveness (Delta Score)
# Represents the average mastery gain across all features. Should be positive.
print(f"Average Mastery Gain per Session: {np.mean(avg_delta_score_history):.4f}")

# R3: Exploration Rate (Demonstrates the balance between exploration and exploitation)
print(f"Final Epsilon (Exploration Rate): {mualo_agent.epsilon:.4f}")

# Save the model weights for deployment on the Google Cloud Function
# mualo_agent.model.save_weights('mualo_dqn_final_weights.h5')


--- RL Training Performance Metrics Summary ---
Final Average Reward (Last 100 episodes): 107.32
Average Mastery Gain per Session: 0.1935
Final Epsilon (Exploration Rate): 0.0097
